# Phase 1 — Dataset Audit and Feature Alignment

Goal: understand what is actually in `data/raw/`, standardize column names without touching the raw files, and determine whether the two datasets can be treated as independent populations before any modeling happens.

Files under audit:
- `PCOS_clinical_1.csv` — 541 rows, 6 columns (infertility-workup subset)
- `PCOS_clinical_2.xlsx` — sheet `Full_new` (541 rows, 44 columns) + sheet `Instructions` (data-collection codebook)
- `PCOS_lifestyle.csv` — 2000 rows, 44 columns

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.data_loader import (
    load_clinical_1,
    load_clinical_2_full,
    load_clinical_2_instructions,
    load_lifestyle,
)
from src.data_validation import (
    audit_dataframe,
    find_non_numeric_in_numeric_like,
    compare_columns,
    check_id_linkage,
    check_patient_duplication,
)
from src.config import TARGET_COL, PATIENT_ID_COL, TABLES_DIR

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## 1. Load the raw data

In [ ]:
clinical_1 = load_clinical_1()
clinical_2 = load_clinical_2_full()
lifestyle = load_lifestyle()
instructions = load_clinical_2_instructions()

print("clinical_1:", clinical_1.shape)
print("clinical_2 (Full_new sheet):", clinical_2.shape)
print("lifestyle:", lifestyle.shape)

In [ ]:
# The 'Instructions' sheet is the original data-collection codebook, not a data table.
# It documents units, the Yes/No -> 1/0 convention, and the Blood Group code map.
instructions

## 2. Structural audit: shape, dtypes, missing values, duplicates, target balance

In [ ]:
for name, df in [("clinical_1", clinical_1), ("clinical_2", clinical_2), ("lifestyle", lifestyle)]:
    audit_dataframe(df, name, target_col=TARGET_COL)

In [ ]:
clinical_1.head()

In [ ]:
clinical_2.head()

In [ ]:
lifestyle.head()

In [ ]:
# Numeric summary statistics — check for implausible min/max values
lifestyle.select_dtypes("number").describe().T

## 3. Potential data-entry errors: text values hiding inside numeric columns

`AMH(ng/mL)` and `II    beta-HCG(mIU/mL)` load as `object` dtype in the raw files instead of `float64`. Before blindly coercing with `pd.to_numeric(..., errors="coerce")`, check exactly which raw values are causing that so nothing is silently dropped without a record of why.

In [ ]:
numeric_like_cols = ["AMH(ng/mL)", "II    beta-HCG(mIU/mL)"]

print("clinical_1:")
print(find_non_numeric_in_numeric_like(clinical_1, numeric_like_cols))
print("\nclinical_2:")
print(find_non_numeric_in_numeric_like(clinical_2, numeric_like_cols))
print("\nlifestyle:")
print(find_non_numeric_in_numeric_like(lifestyle, numeric_like_cols))

## 4. Column-name comparison across the three files

Columns are stripped of surrounding whitespace before comparing, so real naming mismatches (typos) don't get hidden by whitespace differences, but also aren't confused with them.

In [ ]:
col_matrix = compare_columns({
    "clinical_1": clinical_1,
    "clinical_2": clinical_2,
    "lifestyle": lifestyle,
})

# Show only columns that are NOT present in all three, to spot naming inconsistencies
col_matrix[~col_matrix.all(axis=1)]

**Expected finding:** every column in `clinical_2` and `lifestyle` matches except `No. of abortions` (lifestyle) vs `No. of aborptions` (clinical_2 — a typo present in both the raw file and the Instructions codebook). `clinical_1` only carries its own 6 columns.

## 5. Patient linkage: is `clinical_1` describing the same people as `clinical_2`?

`clinical_1`'s `Patient File No.` runs 10001-10541. `clinical_2`'s runs 1-541. Test whether `clinical_1_id - 10000 == clinical_2_id` lines up the same patients (same PCOS label, same AMH, same beta-HCG).

In [ ]:
linkage = check_id_linkage(
    clinical_1, clinical_2,
    id_col_a=PATIENT_ID_COL, id_col_b=PATIENT_ID_COL, id_offset=10000,
    compare_cols=[TARGET_COL, "AMH(ng/mL)"],
)
print("matched rows:", len(linkage), "out of", len(clinical_1), "clinical_1 rows and", len(clinical_2), "clinical_2 rows")
linkage.head(10)

**Expected finding:** all 541 rows match, with identical `PCOS (Y/N)` and `AMH(ng/mL)` values. `clinical_1` is a column-subset extract of the same 541 patients as `clinical_2`, just with a different ID offset. It adds no new patients and, beyond `beta-HCG I/II`, no columns that `clinical_2` doesn't already have.

## 6. The critical check: is `lifestyle` (2000 rows) an independent sample, or derived from the 541-patient file?

This determines whether `clinical_2` / `clinical_1` can be used as genuinely independent external validation data, or whether they are the *source* of `lifestyle` — which would make that framing invalid.

In [ ]:
print("Unique Patient File No. values in lifestyle:", lifestyle[PATIENT_ID_COL].nunique())
print("Range of Patient File No. in lifestyle:", lifestyle[PATIENT_ID_COL].min(), "-", lifestyle[PATIENT_ID_COL].max())
print("Total rows in lifestyle:", len(lifestyle))
print()
print("Repetition count per Patient File No. in lifestyle:")
print(lifestyle[PATIENT_ID_COL].value_counts().describe())

In [ ]:
# Example: every lifestyle row sharing Patient File No. == 1, vs. the single clinical_2 row for that patient
cols_to_show = [" Age (yrs)", "Weight (Kg)", "Height(Cm) ", "BMI", TARGET_COL]

print("clinical_2, Patient File No. == 1:")
print(clinical_2.loc[clinical_2[PATIENT_ID_COL] == 1, cols_to_show])
print("\nlifestyle, all rows with Patient File No. == 1:")
print(lifestyle.loc[lifestyle[PATIENT_ID_COL] == 1, cols_to_show])

In [ ]:
dup_check = check_patient_duplication(
    df_augmented=lifestyle,
    df_base=clinical_2,
    id_col=PATIENT_ID_COL,
    compare_cols=["Weight (Kg)", "Height(Cm) ", "BMI"],
    label_col=TARGET_COL,
)
print("patients with >1 occurrence in lifestyle, matched back to clinical_2:", len(dup_check))
print("any PCOS label disagreement between a patient's repeats and their clinical_2 row:", dup_check["label_mismatch"].any())
dup_check.describe()

**Expected finding:**
- `lifestyle`'s `Patient File No.` only takes values 1-541 (matching `clinical_2`), not 2000 unique IDs.
- Most patient IDs repeat 2-10 times (mean ~3.8) in `lifestyle`.
- For a repeated ID, `Age` and the `PCOS (Y/N)` label are identical across every repeat and match `clinical_2` exactly; continuous measurements (Weight, Height, BMI, ...) differ by a small amount (mean absolute difference roughly 0.5-1.5 units) from the single matching row in `clinical_2`.
- No label disagreements were found across any repeated patient.
- 11 of the 541 patients in `clinical_2` do not appear in `lifestyle` at all.

**Conclusion:** `PCOS_lifestyle.csv` is not an independent 2000-patient sample. It is an expanded/resampled version of the same ~530-541 patients present in `PCOS_clinical_2.xlsx` (and, by extension, `PCOS_clinical_1.csv`), with continuous features perturbed by a small amount per copy and the label held fixed per patient. This means `clinical_1`/`clinical_2` are **not** an independent second population — they are (a subset of) the source data `lifestyle` was generated from.

## 7. Implications for the project design

This changes a load-bearing assumption from the original project plan ("Dataset 1 = 2000 rows for training, Dataset 2 = ~540 rows for independent external validation"):

- **Internal train/test split risk:** a plain random 80/20 split of `lifestyle` can put multiple near-duplicate copies of the *same* patient into both train and test, inflating internal test metrics. Any split of `lifestyle` should be **grouped by `Patient File No.`** (e.g. `GroupShuffleSplit`/`GroupKFold`) so all copies of one patient land on the same side.
- **External validation risk:** using `clinical_1`/`clinical_2` as "held-out external validation" after training on `lifestyle` is not a valid independence test — up to ~530 of those 541 patients contributed directly to the training data. Metrics computed this way would overstate real-world generalization.

This is a design decision, not a coding detail — see the accompanying chat message for options and a recommendation before Phase 2 (EDA) begins, since the choice affects how the data should be split from the very start.

## 8. Feature-mapping table

Generated from the column, unit, and leakage checks above. See `reports/tables/feature_mapping.csv`.

In [ ]:
feature_mapping = pd.read_csv(TABLES_DIR / "feature_mapping.csv")
feature_mapping